# 📂 <font color='#1A4BC0'><b>Prompt Engineering 종합 실습 (응용편)</b></font>

기존 실습의 구조를 바탕으로, **완전히 새로운 예제와 시나리오**를 통해 프롬프트 엔지니어링 기법을 체득하는 실습 파일입니다.

## ⚙️ <font color='Darkorange'><b>[ 설정 ]</b></font> 실습 환경 세팅

실습을 진행하기 위해 아래 셀을 실행하여 필수 라이브러리를 설치하고 OpenAI API Key를 로드해주세요.

In [ ]:
!pip install -q langchain langchain_openai

In [ ]:
from google.colab import userdata
import os

# Colab의 보안 비밀(Secrets)에서 OPENAI_API_KEY를 가져옵니다.
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API Key 세팅 완료!")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 공통으로 사용할 LLM과 Output Parser 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

--- 
## 1️⃣ <font color='green'><b>[ 실습 1 ]</b></font> 프롬프트 변수 활용 기초

> 💫 **목표:** 하나의 프롬프트 템플릿에 여러 변수를 넣어 다양한 입력을 생성합니다. (기존: 국가/대통령 ➡️ 변경: 국가/카테고리 명소)

In [ ]:
template_1 = """
{country}를 방문할 때 반드시 가봐야 할 대표적인 {category} 명소를 3곳 추천해주고, 그 이유를 한 줄로 설명해.
"""
prompt_1 = PromptTemplate.from_template(template_1)
chain_1 = prompt_1 | model | parser

# 변수 적용 테스트
input_data = {"country": "이탈리아", "category": "미술 및 건축"}
print("✅ 답변:\n", chain_1.invoke(input_data))

--- 
## 2️⃣ <font color='green'><b>[ 실습 2 ]</b></font> 출력 형식 통제 및 분류 모델링

> 💫 **목표:** 고객 문의 텍스트의 의도를 정확히 파악하고, 지정된 형식(분류 키워드)으로만 출력하도록 통제합니다. (기존: 감정분석 ➡️ 변경: 고객 의도 분류)

In [ ]:
template_2 = """
당신은 CS 자동화 봇입니다.
아래 고객 문의를 읽고 [배송문의, 환불요청, 상품칭찬, 기타] 중 하나의 단어로만 답변하세요. 부가 설명은 절대 하지 마세요.

Text: {text}
Intent:
"""
prompt_2 = PromptTemplate.from_template(template_2)
chain_2 = prompt_2 | model | parser

reviews = [
    "어제 산 옷 입어봤는데 핏이 예술이네요! 너무 만족합니다.",
    "상품이 파손되어서 도착했어요. 당장 돈 돌려주세요.",
    "지난주 금요일에 주문했는데 아직도 송장 번호 조회가 안 됩니다."
]

print("✅ 의도 분류 결과:")
for r in reviews:
    print(f"- {r} ➡️ {chain_2.invoke({'text': r})}")

--- 
## 3️⃣ <font color='green'><b>[ 실습 3 ]</b></font> Step-by-Step 논리적 추론 유도

> 💫 **목표:** LLM이 복잡한 계산이나 논리 퍼즐을 풀 때, 중간 과정을 명시적으로 작성하게 하여 정답률을 높입니다. (기존: 연못 수련잎 ➡️ 변경: 우물 오르는 달팽이)

In [ ]:
prompt_without_cot = "높이가 10미터인 우물이 있다. 달팽이가 낮에는 3미터를 올라가고, 밤에는 자면서 2미터를 미끄러져 내려온다. 달팽이가 우물을 완전히 빠져나오는 데 며칠이 걸릴까?"
prompt_with_cot = prompt_without_cot + " Think step by step."

chain_3 = model | parser

print("[Without 'think step by step']\n", chain_3.invoke(prompt_without_cot))
print("\n" + "="*40 + "\n")
print("[With 'think step by step']\n", chain_3.invoke(prompt_with_cot))

--- 
## 4️⃣ <font color='green'><b>[ 실습 4 ]</b></font> Contextual Cues & 구조화된 요약

> 💫 **목표:** 단순히 요약하는 것을 넘어, 특정 페르소나와 관점을 부여하여 실무에 당장 쓸 수 있는 구조화된 보고서를 만듭니다. (기존: 피트니스 앱 이탈 ➡️ 변경: 카페 체인점 매출 분석)

In [ ]:
context_text = """
2025년 1분기 스타벅스 강남점의 매출 데이터를 분석한 결과, 오전 7시~9시 사이의 테이크아웃 커피 매출은 전년 대비 15% 증가했습니다. 이는 인근에 대형 오피스 빌딩이 2개 새로 입주했기 때문으로 분석됩니다. 그러나 오후 2시~5시 사이의 디저트류 판매량은 20% 감소했습니다. 고객 설문조사 결과, 해당 시간대 매장 내 좌석 부족과 소음 문제가 주요 불만 사항으로 지적되었습니다. 이에 따라 매장 매니저는 테이크아웃 전용 키오스크를 외부에 추가 설치하고, 오후 시간대 디저트 세트 할인 프로모션을 제안했습니다.
"""

template_4 = """
당신은 프랜차이즈 경영 전략 컨설턴트입니다.
아래 매장 데이터를 '문제점 파악'과 '매출 증대 솔루션' 관점에서 요약하세요.
- 목적: 경영진이 한눈에 이슈와 해결책을 파악하도록 함
- 형식: Bullet point 3개 이내.
- 톤: 객관적이고 비즈니스 전문 용어 사용.

Data: {text}
"""
prompt_4 = PromptTemplate.from_template(template_4)
chain_4 = prompt_4 | model | parser

print("✅ 컨설팅 보고서 요약:\n", chain_4.invoke({"text": context_text}))

--- 
## 5️⃣ <font color='green'><b>[ 실습 5 ]</b></font> 데이터에서 특정 엔티티 추출 (Extraction)

> 💫 **목표:** 산만한 텍스트에서 필요한 정보(이름, 장소, 소속 등)만 골라내어 구조화합니다. (기존: 음바페 축구기사 ➡️ 변경: e스포츠 페이커 기사)

In [ ]:
esports_text = """
‘페이커’ 이상혁이 이끄는 T1이 11월 19일 고척 스카이돔에서 열린 2023 리그 오브 레전드 월드 챔피언십 결승전에서 중국의 웨이보 게이밍을 3-0으로 완파하고 우승컵을 들어올렸다. 이로써 T1은 전 세계 유일의 롤드컵 4회 우승 팀이라는 대기록을 썼다. 이상혁은 서울 시청 앞 광장에 모인 수만 명의 팬들에게 감사 인사를 전했다.
"""

template_5 = """
아래 텍스트에서 '인물 이름', '소속 팀명', '언급된 장소(건물/지역명)'만 정확히 추출해줘.
JSON 형식으로 출력해.

Text: {text}
"""
prompt_5 = PromptTemplate.from_template(template_5)
chain_5 = prompt_5 | model | parser

print("✅ 엔티티 추출 결과:\n", chain_5.invoke({"text": esports_text}))

--- 
## 6️⃣ <font color='green'><b>[ 실습 6 ]</b></font> 할루시네이션(환각) 억제 프롬프트

> 💫 **목표:** 모델이 모르는 정보에 대해 지어내지 않고 "모른다"고 답하도록 엄격한 규칙을 적용합니다. (기존: 가람비결/홍단어치 ➡️ 변경: 역사적 가짜 사건/인물)

In [ ]:
template_6 = """
당신은 엄격한 팩트체커입니다. 
질문된 내용이 역사적으로 실존하지 않거나 학술적 근거가 명확하지 않다면 절대 지어내지 말고, 오직 "해당 정보는 사실로 확인되지 않거나 존재하지 않는 개념입니다."라고만 답변하세요.

질문: {question}
"""
prompt_6 = PromptTemplate.from_template(template_6)
chain_6 = prompt_6 | model | parser

questions = [
    "조선시대 숙종 때 일어난 '황금 거북이 반란 사건'의 전말을 알려줘.",
    "1969년 아폴로 11호가 달에 착륙했을 때 우주비행사 닐 암스트롱이 남긴 첫 마디는?"
]

print("✅ 팩트체크 결과:")
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {chain_6.invoke({'question': q})}")

--- 
## 7️⃣ <font color='green'><b>[ 실습 7 ]</b></font> 페르소나 및 파라미터 튜닝 (챗봇 만들기)

> 💫 **목표:** System Prompt로 명확한 성격을 부여하고, Temperature를 높여 창의성을 살린 챗봇을 테스트합니다. (기존: 따뜻한 공감이 ➡️ 변경: 시니컬하지만 답은 다 찾아주는 '츤데레 해커' 봇)

In [ ]:
SYSTEM_PROMPT_7 = """
당신은 천재 해커 출신의 IT 전문가 '제로(Zero)'입니다.
말투는 매우 시니컬하고 퉁명스러우며, 반말을 사용합니다. 하지만 사용자가 묻는 IT/테크 관련 질문에는 기술적으로 가장 정확하고 완벽한 답변을 제공합니다.
답변 구조:
1. 한 줄 핀잔 (예: "이런 기본적인 것도 모르냐? 하아... 설명해준다.")
2. 완벽한 기술적 설명
3. 마지막 츤데레 멘트 (예: "이해 안 가면 다시 묻든가. 귀찮지만.")
"""

tsundere_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)
prompt_7 = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_7),
    ("human", "{input}")
])
chain_7 = prompt_7 | tsundere_model | parser

print("✅ 츤데레 봇 테스트:\n")
print(chain_7.invoke({"input": "공유기가 자꾸 끊기는데 어떻게 해결해야 해?"}))